# 證件角點偵測（YOLO-pose）訓練 — Google Colab

對應 `id_card_detector/README.md`。這個 notebook 用免費的 Colab GPU（T4）訓練一個小型 YOLO-pose 模型，學習身分證/示範卡片的 4 個角點（左上、右上、右下、左下），取代 `image_utils/id_card.py` 目前的古典 CV 方法。

## 開始之前：先把資料集上傳到 Google Drive

在**本機**（不是 Colab）跑這行，把整個 `dataset/` 資料夾（含 `data.yaml`、`images/`、`labels/`）打包成一個 zip：

```bash
cd id_card_detector
zip -r id_card_dataset.zip dataset/
```

打包完，把 `id_card_dataset.zip` 上傳到你的 Google Drive **根目錄**（`MyDrive/` 底下），再回來繼續跑下面的儲存格。

## 使用方式

上方選單 **執行階段 → 變更執行階段類型 → 硬體加速器選 T4 GPU**，然後從上到下依序執行每個儲存格。

## 1. 掛載 Google Drive、解壓縮資料集

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import zipfile
from pathlib import Path

zip_path = Path('/content/drive/MyDrive/id_card_dataset.zip')
extract_dir = Path('/content/id_card_detector')
extract_dir.mkdir(parents=True, exist_ok=True)

assert zip_path.exists(), f"找不到 {zip_path}——確認 zip 檔真的上傳到 Drive 根目錄了嗎？"

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(extract_dir)

print('解壓縮完成，內容：')
!ls -la /content/id_card_detector/dataset
!echo '---'
!find /content/id_card_detector/dataset/images -type f | wc -l
!find /content/id_card_detector/dataset/labels -type f | wc -l

## 2. 安裝 ultralytics

In [ ]:
!pip install -q ultralytics

import ultralytics
ultralytics.checks()

## 3. 修正 data.yaml 裡的路徑

本機的 `data.yaml` 裡 `path: .` 是相對路徑，在 Colab 這裡要指向解壓縮後的絕對路徑，這裡直接覆寫一份，不用手動編輯上傳的檔案。

In [ ]:
data_yaml_path = Path('/content/id_card_detector/dataset/data.yaml')

data_yaml_content = f"""path: {extract_dir / 'dataset'}
train: images/train
val: images/val

kpt_shape: [4, 3]

names:
  0: id_card
"""
data_yaml_path.write_text(data_yaml_content, encoding='utf-8')
print(data_yaml_path.read_text())

## 4. 訓練

用 `yolo11n-pose`（nano 量級，CPU/小 GPU 都跑得動，之後接回 `image_utils/id_card.py` 時也比較不會拖慢速度）。168 張圖的量不大，`epochs=150` 讓它訓練久一點、搭配 `patience` 提早停止避免 overfitting。

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11n-pose.pt')  # 預訓練權重，micro-tune，不是從零訓練

results = model.train(
    data=str(data_yaml_path),
    epochs=150,
    patience=30,
    imgsz=640,
    batch=16,
    project='/content/runs',
    name='id_card_pose',
)

## 5. 驗證 + 隨便挑幾張看看預測結果

In [ ]:
metrics = model.val(data=str(data_yaml_path))
print(metrics)

In [ ]:
import glob

val_images = glob.glob('/content/id_card_detector/dataset/images/val/*.jpg')[:5]
pred_results = model.predict(val_images, save=True, project='/content/runs', name='id_card_pose_predict')

for r in pred_results:
    print(r.path, '->', r.keypoints.xy if r.keypoints is not None else '沒偵測到')

預測結果的標註圖存在 `/content/runs/id_card_pose_predict/`，可以用左側檔案瀏覽器點開來看，或執行下面這格直接在 notebook 裡顯示。

In [ ]:
from IPython.display import Image, display

for path in sorted(glob.glob('/content/runs/id_card_pose_predict/*.jpg'))[:5]:
    display(Image(filename=path))

## 6. 把訓練好的權重存回 Google Drive

訓練完的最佳權重在 `/content/runs/id_card_pose/weights/best.pt`，複製回 Drive，這樣就算 Colab session 斷線也不會遺失。下載回本機後放進 `id_card_detector/weights/best.pt`。

In [ ]:
import shutil

src = '/content/runs/id_card_pose/weights/best.pt'
dst = '/content/drive/MyDrive/id_card_pose_best.pt'
shutil.copy(src, dst)
print(f'已複製到 {dst}，去 Google Drive 網頁版下載回本機，放進 id_card_detector/weights/best.pt')